# Task 1 — Evaluation Rubric

Deterministic scoring criteria for evaluating model-generated product descriptions.
Each criterion is rated as good, ok, or bad using explicit measurable rules.


## Criterion Definitions

### 1. Fluency (Natural, easy-to-read sentences)

good:
Sentences read naturally with smooth flow and no awkward phrasing.

ok:
Minor awkward phrasing but overall understandable and readable.

bad:
Multiple unnatural or difficult-to-follow sentences that disrupt readability.


### 2. Grammar (Correct spelling & punctuation)

good:
No spelling or punctuation errors.

ok:
1 minor grammar/spelling/punctuation error.

bad:
2 or more grammar/spelling/punctuation errors.


### 3. Tone (Friendly, credible sales voice)

good:
Consistently friendly, engaging, and credible sales tone throughout.

ok:
Mostly appropriate tone with minor inconsistency OR slightly neutral wording.

bad:
Tone inappropriate, overly formal, robotic, or not aligned with sales voice.


### 4. Length (50–90 words)

good:
50–90 words

ok:
40–49 words OR 91–110 words

bad:
Less than 40 words OR more than 110 words


### 5. Grounding (Sticks to provided information)

good:
All claims strictly supported by provided information.

ok:
One minor unsupported detail added.

bad:
Multiple unsupported claims OR contradicts provided information.


### 6. Latency (Average time per call)

good:
Response time within acceptable production threshold (e.g., ≤2 seconds).

ok:
Response time slightly above threshold (2–4 seconds).

bad:
Response time exceeds acceptable limit (>4 seconds).


### 7. Cost (Average price per call per 1K tokens)

good:
Within target budget threshold.

ok:
Slightly above target budget (≤25% over).

bad:
More than 25% above target budget threshold.

## Pass / Fail Definition

### Cumulative pass rule

A response passes if:

- At least 4 criteria are rated good
- No more than 1 criterion is rated bad


### Automatic failure rules (go / no-go)

A response automatically fails if:

- Grounding is not good
OR
- Grammar is bad

# Task 2 — LLM Product description generation

In [2]:
system_prompt = """
You are a professional ecommerce product description writer.

Write concise, engaging product descriptions using:

- product name
- key features
- material
- warranty

Follow these rules:

1. Length: 40–60 words
2. Tone: professional and persuasive
3. Highlight 2–3 key features
4. Mention material if relevant
5. Mention warranty at the end
6. Do NOT invent information
"""

In [2]:
%pip install pandas openpyxl

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 283.0 kB/s eta 0:00:0000:0100:02
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 250.9/250.9 kB 231.3 kB/s eta 0:00:00a 0:00:01
  Using cached numpy-2.2.6-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (16.8 MB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 510.5/510.5 kB 429.1 kB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 348.5/348.5 kB 349.7 kB/s eta 0:00:0000:0100:01

[notice] A new release of pip is available: 23.0.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [17]:
import pandas as pd

df = pd.read_excel("Assignment_01_product_dataset.xlsx")

In [4]:
def format_product(row):
    return f"""
Product name: {row['product_name']}
Attributes: {row['Product_attribute_list']}
Material: {row['material']}
Warranty: {row['warranty']}
"""



In [5]:
format_product(df.iloc[0])

'\nProduct name: Apple iPhone 15 Pro\nAttributes: features: A17 Pro chip, 120\u202fHz ProMotion display, USB‑C fast charging; dimensions: compact\nMaterial: titanium frame, Ceramic Shield glass\nWarranty: 1‑year limited warranty\n'

In [6]:
%pip install python-dotenv openai


  Using cached python_dotenv-1.2.2-py3-none-any.whl (22 kB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 419.5 kB/s eta 0:00:0000:0100:01
  Using cached pydantic-2.12.5-py3-none-any.whl (463 kB)
  Using cached httpx-0.28.1-py3-none-any.whl (73 kB)
  Using cached sniffio-1.3.1-py3-none-any.whl (10 kB)
  Using cached tqdm-4.67.3-py3-none-any.whl (78 kB)
  Using cached jiter-0.13.0-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (362 kB)
  Using cached distro-1.9.0-py3-none-any.whl (20 kB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.4/114.4 kB 344.3 kB/s eta 0:00:0000:0100:01
  Using cached idna-3.11-py3-none-any.whl (71 kB)
  Using cached httpcore-1.0.9-py3-none-any.whl (78 kB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 153.7/153.7 kB 311.5 kB/s eta 0:00:0000:0100:01
  Using cached h11-0.16.0-py3-none-any.whl (37 kB)
  Using cached annotated_types-0.7.0-py3-none-any.whl (13 kB)
  Using cached typing_inspection-0.4.2-py3-none-any.whl (14 kB)
  Using cached pydant

In [11]:
import os
from openai import OpenAI
from dotenv import load_dotenv
load_dotenv()

client = OpenAI(
    base_url="https://api.tokenfactory.nebius.com/v1/",
    api_key=os.environ.get("NEBIUS_API_KEY")
)



In [14]:
import time

results = []

for _, row in df.iterrows():

    product_prompt = format_product(row)

    start_time = time.time()

    response = client.chat.completions.create(
        model="meta-llama/Meta-Llama-3.1-8B-Instruct",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": product_prompt}
        ]
    )

    end_time = time.time()

    latency_ms = (end_time - start_time) * 1000

    results.append({
        "description": response.choices[0].message.content,
        "latency_ms": latency_ms,
        "input_tokens": response.usage.prompt_tokens,
        "output_tokens": response.usage.completion_tokens
    })
    
metrics_df = pd.DataFrame(results)


In [16]:
metrics_df.tail()

,description,latency_ms,input_tokens,output_tokens
45,Capture life's moments with the SanDisk Extre...,2251.805067,149,73
46,"""Take your gaming performance to the next leve...",1587.749481,153,69
47,Elevate your gaming experience with the Acer P...,2099.217176,155,77
48,"""Experience breathtaking visuals with the BenQ...",1775.485039,153,74
49,"Here's a concise product description:\n\n""Unlo...",1790.682316,153,80


In [19]:
df = pd.concat([df, metrics_df], axis=1)

In [21]:
rubric_columns = [
    "fluency_score",
    "grammar_score",
    "tone_score",
    "length_score",
    "grounding_score",
    "latency_score",
    "cost_score"
]

for col in rubric_columns:
    df[col] = ""

In [25]:
df["final_score"] = ""


In [26]:
df.to_excel("assignment_01.xlsx", index=False)